# 01 · Baseline: TF-IDF + Regresión Logística

**InterviewIQ · AD5018 · UTEC 2026-2**

Clasificación multilabel del método STAR: un clasificador binario por componente (Situación, Tarea, Acción, Resultado). Este notebook entrena el **baseline** (TF-IDF + Logistic Regression) con umbral fijo 0.5. El modelo final (Linear SVM) está en `02_modelo_svm.ipynb`.

## 1. Datos y split

In [1]:
import re, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEMILLA = 42
ETIQUETAS = ["situacion", "tarea", "accion", "resultado"]

def limpiar(t):
    # Misma limpieza que usa app.py antes de predecir
    t = str(t).lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df = pd.read_csv(RAIZ / "data/raw/interviewiq_star_v1.csv")
df["texto"] = df["respuesta"].map(limpiar)

# Split 70 / 10 / 20 (entrenamiento / validación / prueba)
idx_tmp, idx_test = train_test_split(df.index, test_size=0.20, random_state=SEMILLA)
idx_train, idx_val = train_test_split(idx_tmp, test_size=0.125, random_state=SEMILLA)  # 0.125 * 0.8 = 0.10
train, val, test = df.loc[idx_train], df.loc[idx_val], df.loc[idx_test]
print(f"Total: {len(df)} | train: {len(train)} | val: {len(val)} | test: {len(test)}")
df[ETIQUETAS].mean().rename("% positivos").round(2).to_frame().T

Total: 200 | train: 140 | val: 20 | test: 40


,situacion,tarea,accion,resultado
% positivos,0.66,0.55,0.68,0.38


## 2. Entrenamiento del baseline

Mismo vectorizador que el modelo final para que la comparación sea justa: TF-IDF con unigramas y bigramas, `max_features=5000`, `min_df=2`, `sublinear_tf=True`.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def pipeline_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, min_df=2, ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEMILLA)),
    ])

filas = []
for et in ETIQUETAS:
    m = pipeline_lr().fit(train["texto"], train[et])
    fila = {"componente": et}
    for nombre, parte in [("val", val), ("test", test)]:
        pred = (m.predict_proba(parte["texto"])[:, 1] >= 0.5).astype(int)
        fila[f"F1 macro {nombre}"] = f1_score(parte[et], pred, average="macro")
    filas.append(fila)

res_lr = pd.DataFrame(filas).set_index("componente")
res_lr.loc["PROMEDIO"] = res_lr.mean()
res_lr.round(3)

,F1 macro val,F1 macro test
componente,,
situacion,0.943,0.971
tarea,0.792,0.893
accion,0.867,0.796
resultado,0.798,0.824
PROMEDIO,0.850,0.871


In [3]:
(RAIZ / "notebooks").mkdir(exist_ok=True)
res_lr.round(3).to_csv(RAIZ / "notebooks/resultados_baseline_lr.csv")
print("Guardado en notebooks/resultados_baseline_lr.csv")

Guardado en notebooks/resultados_baseline_lr.csv


## 3. Conclusión

El baseline sirve como piso de referencia. En `02_modelo_svm.ipynb` se compara contra TF-IDF + Linear SVM, se ajustan los umbrales por componente y se analizan las matrices de confusión.